# EXP5 — Wine Clustering + Supervised from Clusters
This notebook reproduces the steps in `EXP5.py`: load the `wine-clustering.csv` dataset, scale features, run K-Means (k=3), create supervised labels from clusters, train a Random Forest classifier on those labels, evaluate, and save visualizations and the labeled CSV.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Helper to save the two visualizations (elbow + PCA cluster plot)
def save_kmeans_visualizations(X_scaled, cluster_labels, kmeans, output_dir):
    k_values = range(1, 11)
    inertias = []
    for k in k_values:
        model = KMeans(n_clusters=k, random_state=42, n_init=10)
        model.fit(X_scaled)
        inertias.append(model.inertia_)

    plt.figure(figsize=(8, 5))
    plt.plot(k_values, inertias, marker="o")
    plt.title("Elbow Method for K-Means")
    plt.xlabel("Number of Clusters (k)")
    plt.ylabel("Inertia")
    plt.grid(alpha=0.3)
    elbow_path = output_dir / "kmeans_elbow_curve.png"
    plt.tight_layout()
    plt.savefig(elbow_path, dpi=150)
    plt.close()

    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X_scaled)
    centers_pca = pca.transform(kmeans.cluster_centers_)

    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(
        X_pca[:, 0],
        X_pca[:, 1],
        c=cluster_labels,
        cmap="viridis",
        alpha=0.75,
        edgecolors="k",
        s=45,
    )
    plt.scatter(
        centers_pca[:, 0],
        centers_pca[:, 1],
        c="red",
        marker="X",
        s=220,
        label="Centroids",
    )
    plt.title("K-Means Clusters Visualized with PCA (2D)")
    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.legend(loc="best")
    plt.colorbar(scatter, label="Cluster ID (0-based)")
    plt.grid(alpha=0.2)
    cluster_plot_path = output_dir / "kmeans_clusters_pca.png"
    plt.tight_layout()
    plt.savefig(cluster_plot_path, dpi=150)
    plt.close()

    return elbow_path, cluster_plot_path

In [ ]:
# 1) Load data
base_dir = Path('.')
file_path = base_dir / 'wine-clustering.csv'
df = pd.read_csv(file_path)

# 2) Prepare features
feature_columns = df.columns.tolist()
X = df[feature_columns].copy()

# 3) Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4) KMeans clustering (k=3)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)
df['Wine_Type'] = cluster_labels + 1

# 5) Build supervised dataset from clusters
X_supervised = X_scaled
y_supervised = df['Wine_Type']

X_train, X_test, y_train, y_test = train_test_split(
    X_supervised, y_supervised, test_size=0.25, random_state=42, stratify=y_supervised
)

# 6) Train Random Forest classifier
clf = RandomForestClassifier(n_estimators=300, random_state=42)
clf.fit(X_train, y_train)

# 7) Evaluate
y_pred = clf.predict(X_test)
print('=== Supervised Learning on Generated Wine Types ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print('
Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))
print('
Classification Report:')
print(classification_report(y_test, y_pred, digits=4))

# 8) Cluster profile
print('
=== Mean Feature Values Per Discovered Wine Type ===')
cluster_profile = df.groupby('Wine_Type')[feature_columns].mean()
display(cluster_profile)

# 9) Save visualizations and labeled CSV
elbow_path, cluster_plot_path = save_kmeans_visualizations(X_scaled, cluster_labels, kmeans, base_dir)
print('
Saved visualizations:')
print('-', elbow_path)
print('-', cluster_plot_path)
output_path = base_dir / 'wine-supervised.csv'
df.to_csv(output_path, index=False)
print(f'
Saved supervised dataset with labels to: {output_path}')